## Spring Thaw Accessor

### Fetching and Visualizing Steamboat Springs, CO boundary
Grabbing boundary from OSM, checking and obtaining type, bbox and doing a quick visualization.

In [13]:
import osmnx
import geopandas

# Grabbing boundary from OSM
gdf = osmnx.geocoder.geocode_to_gdf("Steamboat Springs, Colorado") # Fetches Steamboat Springs, CO boundary into a gdf

# Checking geometry type of gdf, and obtaining that geometry
print(gdf.geom_type)
geom = gdf.geometry[0]
print(geom)
print(f'GDF CRS is: ', {gdf.crs})

# Obtain bbox from gdf
bounds = gdf.bounds
minx = bounds.iloc[0]['minx']
miny = bounds.iloc[0]['miny']
maxx = bounds.iloc[0]['maxx']
maxy = bounds.iloc[0]['maxy']

print(f"Bounding box for Steamboat Springs, CO boundary: {minx}, {miny}, {maxx}, {maxy}")

# gdf.geometry[0] #quick viz
m = gdf.explore()
m

0    Polygon
dtype: object
POLYGON ((-106.886846 40.507395, -106.886418 40.506932, -106.886018 40.506654, -106.885778 40.506473, -106.885349 40.506142, -106.885046 40.506031, -106.884879 40.50597, -106.883096 40.506038, -106.882489 40.506211, -106.882049 40.506337, -106.881803 40.506013, -106.881505 40.505329, -106.881439 40.505382, -106.88105 40.505704, -106.881037 40.505715, -106.880977 40.505678, -106.880711 40.505514, -106.880399 40.505323, -106.879631 40.50505, -106.879248 40.504913, -106.879041 40.50484, -106.878912 40.504807, -106.878639 40.504737, -106.87845 40.504688, -106.878465 40.504637, -106.878476 40.504601, -106.878474 40.50452, -106.878498 40.504488, -106.878531 40.504447, -106.878508 40.504334, -106.878497 40.504279, -106.878486 40.50422, -106.878406 40.503821, -106.878397 40.503777, -106.878385 40.503716, -106.878364 40.50361, -106.878352 40.503559, -106.878333 40.503474, -106.878266 40.503453, -106.878217 40.503438, -106.878134 40.50343, -106.878051 40.503423, -106.8

### Downloading SNODAS Raster
Download, reproject (if needed), clip raster and create vector grid. 
SNODAS raster is in .tar format, and inside is a .dat.gz folder.

In [21]:
import shutil, tarfile, gzip, urllib.request, os

# Variable Setup
target_dir = os.path.join("data", "raw")
if not os.path.exists(target_dir):
    os.makedirs(target_dir)

url = 'https://noaadata.apps.nsidc.org/NOAA/G02158/masked/2024/04_Apr/SNODAS_20240412.tar'
tar_filepath = os.path.join(target_dir, 'SNODAS_20240412.tar')
tar_pre = 'us_ssmv11034'
tar_suf = '.dat.gz'

print("Starting download...")

# Download tar file from URL
urllib.request.urlretrieve(url, tar_filepath)

# Open .tar file to peek inside
with tarfile.open(tar_filepath, 'r') as tar:
    # Get list of all files inside
    all_files = tar.getnames()

    found_file = None

    # Look for the specific .dat.gz file
    for name in all_files:
        if name.startswith(tar_pre) and name.endswith(tar_suf):
            found_file = name
            print('Target file found. Extracting...')
            # Extract the file
            tar.extract(found_file, path=target_dir)
            break

# Unzip .gz part to get the .dat
if found_file:
    gz_full_path = os.path.join(target_dir, found_file)
    output_dat_file = gz_full_path.replace(".gz", "")
    
    print(f"Unzipping to {output_dat_file}...")

    with gzip.open(gz_full_path, 'rb') as f_in:
        with open(output_dat_file, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)

    print(".dat file is ready")
else:
    print("Could not find file")


Starting download...
Target file found. Extracting...
Unzipping to data\raw\us_ssmv11034tS__T0001TTNATS2024041205HP001.dat...
.dat file is ready


C:\Users\DJ\AppData\Local\Temp\ipykernel_14500\2127196138.py:31: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(found_file, path=target_dir)


### Converting SNODAS to GeoTiff


In [ ]:
from osgeo import gdal, osr
import numpy as np



sno_file = 'snowdas_20240412.tif'



def convert_snodas_to_tif(input_dat, output_tif):
    
    output_processed = 'data/processed'
    os.makedirs(output_processed, exist_ok=True)

    snow_target_dir = os.path.join(output_processed, output_tif)
    
    # Grid size
    cols = 6935
    rows = 3351

    # Read the binary data
    raw_data = np.fromfile(input_dat, dtype='>i2').reshape((rows,cols))

    # Create the GeoTIFF driver
    driver = gdal.GetDriverByName('GTiff')
    dataset = driver.Create(snow_target_dir, cols, rows, 1, gdal.GDT_Int16)

    # Define Geotransform
    dataset.SetGeoTransform([-124.73333333333333, 0.00833333333333, 0, 
                              52.87500000000000, 0, -0.00833333333333])

    # Define Projection
    srs = osr.SpatialReference()
    srs.ImportFromEPSG(4326)
    dataset.SetProjection(srs.ExportToWkt())

    # Write data to band 1
    band = dataset.GetRasterBand(1)
    band.WriteArray(raw_data)
    band.SetNoDataValue(-9999)

    # Flush to disk
    dataset = None
    print(f"Successfully created {snow_target_dir}")

# Usage
convert_snodas_to_tif(output_dat_file, sno_file)


Successfully created snowdas_20240412.tif
